# 01 Data Gathering - 8 Dataset Terpisah

Notebook ini hanya membaca 8 file raw dari `data/raw/` dan menampilkan hasilnya di output cell.


## Setup Environment
Import library dipisahkan dari konfigurasi path dan function agar alur notebook mudah dibaca dan dipakai ulang.


In [1]:
from pathlib import Path
from zipfile import ZipFile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

try:
    from fingo_ds1.config import RAW_DATA_PATH
except ImportError:
    RAW_DATA_PATH = None


## Konfigurasi Path dan Tampilan
Sel ini hanya menyiapkan lokasi project, lokasi data raw, dan opsi tampilan notebook.


In [2]:
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

if RAW_DATA_PATH is None:
    RAW_DATA_PATH = project_root / 'data' / 'raw'

raw_data_path = Path(RAW_DATA_PATH)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
sns.set_theme(style='whitegrid', palette='Set2')

print(f'Project root  : {project_root}')
print(f'Raw data path : {raw_data_path}')
print('Mode output   : display-only, tidak menyimpan file')


Project root  : /home/umaygans/05_nayyara_submission_1/nayyara_capstone
Raw data path : /home/umaygans/05_nayyara_submission_1/nayyara_capstone/data/raw
Mode output   : display-only, tidak menyimpan file


## Dataset Catalog
Delapan dataset didefinisikan eksplisit berdasarkan isi folder `data/raw/`. Dataset e-commerce bulanan tetap diperlakukan sebagai 6 dataset terpisah.


In [3]:
DATASET_CATALOG = [
    {
        'dataset_id': 'ecommerce_2024_01_january',
        'dataset_name': 'E-Commerce Sales - January 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-01',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '01_JanuarySales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_06_june',
        'dataset_name': 'E-Commerce Sales - June 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-06',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '02_JuneSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2024_12_december',
        'dataset_name': 'E-Commerce Sales - December 2024',
        'domain': 'ecommerce_sales',
        'dataset_period': '2024-12',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '03_DecemberSales2024_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_02_february',
        'dataset_name': 'E-Commerce Sales - February 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-02',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '04_FebruarySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_07_july',
        'dataset_name': 'E-Commerce Sales - July 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-07',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '05_JulySales2025_clean.xlsx',
    },
    {
        'dataset_id': 'ecommerce_2025_11_november',
        'dataset_name': 'E-Commerce Sales - November 2025',
        'domain': 'ecommerce_sales',
        'dataset_period': '2025-11',
        'source_path': raw_data_path / 'Indonesian_Ecommerce_sales' / '06_NovemberSales2025_clean.xlsx',
    },
    {
        'dataset_id': 'daily_household_transactions',
        'dataset_name': 'Daily Household Transactions',
        'domain': 'household_finance',
        'dataset_period': None,
        'source_path': raw_data_path / 'daily_household_transaction' / 'Daily Household Transactions.csv',
    },
    {
        'dataset_id': 'personal_finance',
        'dataset_name': 'Personal Finance Dataset',
        'domain': 'personal_finance',
        'dataset_period': None,
        'source_path': raw_data_path / 'personal_finance' / 'Personal_Finance_Dataset.csv',
    },
]

assert len(DATASET_CATALOG) == 8, 'Catalog harus berisi tepat 8 dataset.'


## Reusable Loader
Helper ini membaca CSV dan Excel. Jika `openpyxl` belum tersedia, file `.xlsx` tetap bisa dibaca memakai fallback XML dari struktur zip Excel.


In [4]:
SUPPORTED_EXTENSIONS = {'.csv', '.xlsx', '.xls'}
EXCEL_NAMESPACE = {'main': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}


def make_unique_columns(columns):
    counts = {}
    clean_columns = []

    for column in columns:
        text = '' if pd.isna(column) else str(column).strip()
        text = text or 'unnamed_column'
        counts[text] = counts.get(text, 0) + 1
        clean_columns.append(text if counts[text] == 1 else f'{text}_{counts[text]}')

    return clean_columns


def excel_column_index(cell_reference):
    letters = ''.join(character for character in str(cell_reference) if character.isalpha()) or 'A'
    index = 0

    for letter in letters.upper():
        index = index * 26 + ord(letter) - ord('A') + 1

    return index - 1


def read_excel_without_engine(file_path):
    with ZipFile(file_path) as archive:
        names = set(archive.namelist())
        shared_strings = []

        if 'xl/sharedStrings.xml' in names:
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            shared_strings = [
                ''.join(text.text or '' for text in item.findall('.//main:t', EXCEL_NAMESPACE))
                for item in root.findall('main:si', EXCEL_NAMESPACE)
            ]

        worksheet_names = sorted(name for name in names if name.startswith('xl/worksheets/sheet') and name.endswith('.xml'))
        if not worksheet_names:
            return pd.DataFrame()

        sheet = ET.fromstring(archive.read(worksheet_names[0]))
        rows = []
        width = 0

        for row in sheet.findall('.//main:sheetData/main:row', EXCEL_NAMESPACE):
            values = []
            for cell in row.findall('main:c', EXCEL_NAMESPACE):
                column_index = excel_column_index(cell.attrib.get('r', 'A1'))
                while len(values) <= column_index:
                    values.append(np.nan)

                cell_type = cell.attrib.get('t')
                if cell_type == 'inlineStr':
                    value = ''.join(text.text or '' for text in cell.findall('.//main:t', EXCEL_NAMESPACE))
                else:
                    raw_value = cell.find('main:v', EXCEL_NAMESPACE)
                    value = raw_value.text if raw_value is not None else np.nan
                    if cell_type == 's' and pd.notna(value):
                        value = shared_strings[int(value)]

                values[column_index] = value

            width = max(width, len(values))
            rows.append(values)

    if not rows:
        return pd.DataFrame()

    rows = [row + [np.nan] * (width - len(row)) for row in rows]
    columns = make_unique_columns(rows[0])
    return pd.DataFrame(rows[1:], columns=columns).replace(r'^\s*$', np.nan, regex=True)


def read_data_file(file_path):
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix not in SUPPORTED_EXTENSIONS:
        raise ValueError(f'Unsupported file extension: {file_path.suffix}')

    if suffix == '.csv':
        return pd.read_csv(file_path, sep=None, engine='python', encoding='utf-8-sig').replace(r'^\s*$', np.nan, regex=True)

    try:
        return pd.read_excel(file_path).replace(r'^\s*$', np.nan, regex=True)
    except ImportError:
        return read_excel_without_engine(file_path)


def relative_to_project(path):
    path = Path(path)
    try:
        return str(path.relative_to(project_root))
    except ValueError:
        return str(path)


## Validasi Dataset Catalog
Preview catalog dibuat dengan helper agar pengecekan file bisa dipakai ulang jika catalog berubah.


In [5]:
def build_catalog_preview(catalog):
    return pd.DataFrame(
        {
            'dataset_id': item['dataset_id'],
            'dataset_name': item['dataset_name'],
            'domain': item['domain'],
            'dataset_period': item['dataset_period'],
            'source_path': relative_to_project(item['source_path']),
            'file_exists': item['source_path'].exists(),
            'file_size_kb': round(item['source_path'].stat().st_size / 1024, 2) if item['source_path'].exists() else np.nan,
        }
        for item in catalog
    )


catalog_preview = build_catalog_preview(DATASET_CATALOG)
display(catalog_preview)

missing_files = catalog_preview.loc[~catalog_preview['file_exists'], 'source_path'].tolist()
if missing_files:
    raise FileNotFoundError(f'File raw tidak ditemukan: {missing_files}')


,dataset_id,dataset_name,domain,dataset_period,source_path,file_exists,file_size_kb
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,True,42.92
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,True,65.45
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,True,101.40
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,True,89.67
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,True,65.19
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,True,103.07
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,True,185.50
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,True,89.40


## Load Semua Dataset Secara Terpisah
Function `load_raw_datasets()` menjaga proses load tetap reusable dan tidak menggabungkan transaksi mentah.


In [6]:
def attach_source_metadata(frame, meta):
    dataset_id = meta['dataset_id']
    enriched = frame.copy()
    enriched['_dataset_id'] = dataset_id
    enriched['_dataset_name'] = meta['dataset_name']
    enriched['_domain'] = meta['domain']
    enriched['_dataset_period'] = meta['dataset_period']
    enriched['_source_file'] = meta['source_path'].name
    enriched['_source_path'] = relative_to_project(meta['source_path'])
    return enriched


def build_manifest_row(frame, meta):
    return {
        'dataset_id': meta['dataset_id'],
        'dataset_name': meta['dataset_name'],
        'domain': meta['domain'],
        'dataset_period': meta['dataset_period'],
        'source_path': relative_to_project(meta['source_path']),
        'record_count': int(len(frame)),
        'column_count': int(frame.shape[1]),
    }


def load_raw_datasets(catalog):
    datasets = {}
    manifest_rows = []

    for meta in catalog:
        dataset_id = meta['dataset_id']
        frame = attach_source_metadata(read_data_file(meta['source_path']), meta)
        datasets[dataset_id] = frame
        manifest_rows.append(build_manifest_row(frame, meta))
        print(f'{dataset_id}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns')

    manifest = pd.DataFrame(manifest_rows)
    reusable_catalog = [{**meta, 'source_path': relative_to_project(meta['source_path'])} for meta in catalog]
    return datasets, manifest, reusable_catalog


In [7]:
raw_datasets, manifest_df, dataset_catalog = load_raw_datasets(DATASET_CATALOG)
display(manifest_df)


ecommerce_2024_01_january: 431 rows x 24 columns
ecommerce_2024_06_june: 697 rows x 24 columns


ecommerce_2024_12_december: 1,214 rows x 23 columns


ecommerce_2025_02_february: 957 rows x 24 columns
ecommerce_2025_07_july: 766 rows x 23 columns


ecommerce_2025_11_november: 1,131 rows x 24 columns
daily_household_transactions: 2,461 rows x 14 columns
personal_finance: 1,500 rows x 11 columns


,dataset_id,dataset_name,domain,dataset_period,source_path,record_count,column_count
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,431,24
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,697,24
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,1214,23
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,957,24
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,766,23
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,1131,24
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,2461,14
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,1500,11


## Preview Struktur Tiap Dataset
Preview singkat ini membantu memastikan schema tiap dataset tetap terlihat sebelum masuk tahap assessing.


In [8]:
for dataset_id, frame in raw_datasets.items():
    print('\n' + '=' * 100)
    print(dataset_id)
    print(f'Shape: {frame.shape}')
    print('Columns:')
    print(list(frame.columns))
    display(frame.head(3))



ecommerce_2024_01_january
Shape: (431, 24)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0006556,2,50,0,0,Aksesoris ID Card,1,Selesai,NaN,Hemat Kargo-SPX Hemat,Saldo ShopeePay,KAB. BEKASI,JAWA BARAT,0,8000,10000,8000,2024-01-01 00:05,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...
1,ORD_0006557,2,1000,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA BANDUNG,JAWA BARAT,0,10000,35663,10000,2024-01-01 00:07,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...
2,ORD_0006558,1,600,0,0,Plastik / Wadah Plastik,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,COD (Bayar di Tempat),KAB. BOGOR,JAWA BARAT,0,10000,23187,10000,2024-01-01 00:07,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,01_JanuarySales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/01_January...



ecommerce_2024_06_june
Shape: (697, 24)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', 'Waktu Pesanan Dibuat', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0010173,4,4000,0,0,Other,1,Selesai,NaN,Kargo-J&T Cargo,Kartu Kredit/Debit,KOTA MEDAN,SUMATERA UTARA,25000,20000,242000,45000,2024-06-01 03:22,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...
1,ORD_0010174,3,1500,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-JNE Reguler,Online Payment,KOTA BEKASI,JAWA BARAT,10000,10000,72082,20000,2024-06-01 06:05,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...
2,ORD_0010175,1,500,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-JNE Reguler,SPayLater,KOTA JAKARTA TIMUR,DKI JAKARTA,0,10000,18069,10000,2024-06-01 06:26,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,02_JuneSales2024_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...



ecommerce_2024_12_december
Shape: (1214, 23)
Columns:
['order_id', 'total_qty', 'total_weight_gr', 'total_returned_qty', 'Total Diskon', 'product_categories', 'num_product_categories', 'Status Pesanan', 'Alasan Pembatalan', 'Opsi Pengiriman', 'Metode Pembayaran', 'Kota/Kabupaten', 'Provinsi', 'Ongkos Kirim Dibayar oleh Pembeli', 'Estimasi Potongan Biaya Pengiriman', 'Total Pembayaran', 'Perkiraan Ongkos Kirim', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


...

,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,ORD_0015042,1,500,0,0,Celengan,1,Selesai,NaN,SPX Standard,SPayLater,KOTA PALEMBANG,SUMATERA SELATAN,0,9900,18900,9900,2025-11-01 06:08,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...
1,ORD_0015043,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA MEDAN,SUMATERA UTARA,0,17000,31990,17000,2025-11-01 06:16,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...
2,ORD_0015044,1,6000,0,0,Nampan / Tray,1,Selesai,NaN,J&T Cargo,Saldo ShopeePay,KAB. WAY KANAN,LAMPUNG,0,54493,126900,54493,2025-11-01 06:41,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,06_NovemberSales2025_clean.xlsx,data/raw/Indonesian_Ecommerce_sales/06_Novembe...



daily_household_transactions
Shape: (2461, 14)
Columns:
['Date', 'Mode', 'Category', 'Subcategory', 'Note', 'Amount', 'Income/Expense', 'Currency', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,Date,Mode,Category,Subcategory,Note,Amount,Income/Expense,Currency,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,20/09/2018 12:04:08,Cash,Transportation,Train,2 Place 5 to Place 0,30.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...
1,20/09/2018 12:03:15,Cash,Food,snacks,Idli medu Vada mix 2 plates,60.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...
2,19/09/2018,Saving Bank account 1,subscription,Netflix,1 month subscription,199.0,Expense,INR,daily_household_transactions,Daily Household Transactions,household_finance,None,Daily Household Transactions.csv,data/raw/daily_household_transaction/Daily Hou...



personal_finance
Shape: (1500, 11)
Columns:
['Date', 'Transaction Description', 'Category', 'Amount', 'Type', '_dataset_id', '_dataset_name', '_domain', '_dataset_period', '_source_file', '_source_path']


,Date,Transaction Description,Category,Amount,Type,_dataset_id,_dataset_name,_domain,_dataset_period,_source_file,_source_path
0,2020-01-02,Score each.,Food & Drink,1485.69,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...
1,2020-01-02,Quality throughout.,Utilities,1475.58,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...
2,2020-01-04,Instead ahead despite measure ago.,Rent,1185.08,Expense,personal_finance,Personal Finance Dataset,personal_finance,None,Personal_Finance_Dataset.csv,data/raw/personal_finance/Personal_Finance_Dat...


## Ringkasan 
Dataset dan manifest hanya disimpan di variabel notebook (`raw_datasets`, `manifest_df`, `dataset_catalog`). Tidak ada file output yang dibuat.


In [9]:
print('Dataset tersedia di memory:')
for dataset_id, frame in raw_datasets.items():
    print(f'- {dataset_id}: {len(frame):,} rows')

print('\nTidak ada file yang disimpan dari tahap gathering.')
display(manifest_df)


Dataset tersedia di memory:
- ecommerce_2024_01_january: 431 rows
- ecommerce_2024_06_june: 697 rows
- ecommerce_2024_12_december: 1,214 rows
- ecommerce_2025_02_february: 957 rows
- ecommerce_2025_07_july: 766 rows
- ecommerce_2025_11_november: 1,131 rows
- daily_household_transactions: 2,461 rows
- personal_finance: 1,500 rows

Tidak ada file yang disimpan dari tahap gathering.


,dataset_id,dataset_name,domain,dataset_period,source_path,record_count,column_count
0,ecommerce_2024_01_january,E-Commerce Sales - January 2024,ecommerce_sales,2024-01,data/raw/Indonesian_Ecommerce_sales/01_January...,431,24
1,ecommerce_2024_06_june,E-Commerce Sales - June 2024,ecommerce_sales,2024-06,data/raw/Indonesian_Ecommerce_sales/02_JuneSal...,697,24
2,ecommerce_2024_12_december,E-Commerce Sales - December 2024,ecommerce_sales,2024-12,data/raw/Indonesian_Ecommerce_sales/03_Decembe...,1214,23
3,ecommerce_2025_02_february,E-Commerce Sales - February 2025,ecommerce_sales,2025-02,data/raw/Indonesian_Ecommerce_sales/04_Februar...,957,24
4,ecommerce_2025_07_july,E-Commerce Sales - July 2025,ecommerce_sales,2025-07,data/raw/Indonesian_Ecommerce_sales/05_JulySal...,766,23
5,ecommerce_2025_11_november,E-Commerce Sales - November 2025,ecommerce_sales,2025-11,data/raw/Indonesian_Ecommerce_sales/06_Novembe...,1131,24
6,daily_household_transactions,Daily Household Transactions,household_finance,None,data/raw/daily_household_transaction/Daily Hou...,2461,14
7,personal_finance,Personal Finance Dataset,personal_finance,None,data/raw/personal_finance/Personal_Finance_Dat...,1500,11


## Visualisasi Ringkas Hasil Gathering
Visualisasi memakai `seaborn` dan metadata per dataset, bukan gabungan transaksi mentah.


In [10]:
def build_gathering_plot_frames(manifest):
    summary = manifest.sort_values('record_count', ascending=True).copy()
    summary['short_name'] = summary['dataset_id'].str.replace('ecommerce_', 'ecom_', regex=False)
    domain = (
        manifest['domain']
        .value_counts()
        .sort_values(ascending=True)
        .rename_axis('domain')
        .reset_index(name='dataset_count')
    )
    return summary, domain


def label_horizontal_bars(axis, formatter):
    for container in axis.containers:
        labels = [formatter(patch.get_width()) for patch in container]
        axis.bar_label(container, labels=labels, padding=3)


def plot_gathering_summary(manifest):
    summary, domain = build_gathering_plot_frames(manifest)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

    sns.barplot(data=summary, x='record_count', y='short_name', ax=axes[0], color='#2f6f73')
    axes[0].set_title('Jumlah Baris per Dataset')
    axes[0].set_xlabel('Jumlah baris')
    axes[0].set_ylabel('Dataset')
    label_horizontal_bars(axes[0], lambda value: f'{value:,.0f}')

    sns.barplot(data=summary, x='column_count', y='short_name', ax=axes[1], color='#bf7b45')
    axes[1].set_title('Jumlah Kolom per Dataset')
    axes[1].set_xlabel('Jumlah kolom')
    axes[1].set_ylabel('')
    label_horizontal_bars(axes[1], lambda value: f'{value:,.0f}')

    sns.barplot(data=domain, x='dataset_count', y='domain', ax=axes[2], color='#6f5f90')
    axes[2].set_title('Jumlah Dataset per Domain')
    axes[2].set_xlabel('Jumlah dataset')
    axes[2].set_ylabel('Domain')
    label_horizontal_bars(axes[2], lambda value: f'{value:,.0f}')

    for axis in axes:
        axis.grid(axis='x', linestyle='--', alpha=0.35)
        axis.grid(axis='y', visible=False)
        sns.despine(ax=axis, left=True, bottom=False)

    plt.show()


plot_gathering_summary(manifest_df)
print('Visualisasi seaborn ditampilkan inline; tidak disimpan ke file.')


Visualisasi seaborn ditampilkan inline; tidak disimpan ke file.


## Output Gathering
Tahap gathering selesai ketika 8 dataset sudah terbaca dan tampil di notebook. Untuk tahap berikutnya, jalankan notebook secara berurutan atau gunakan loader raw yang sudah tersedia di notebook berikutnya.


## Area Analisis Mandiri
Gunakan cell kosong di bawah untuk eksplorasi sendiri setelah semua cell utama dijalankan.

Function dan variabel yang bisa dipakai ulang:
- `read_data_file(file_path)`: membaca file CSV/XLSX menjadi dataframe.
- `load_raw_datasets(DATASET_CATALOG)`: membaca semua dataset raw secara terpisah dan membuat manifest.
- `build_catalog_preview(DATASET_CATALOG)`: mengecek path, status file, dan ukuran file raw.
- `plot_gathering_summary(manifest_df)`: membuat visualisasi ringkas hasil gathering dengan seaborn.
- `raw_datasets`: dictionary berisi dataframe mentah per `dataset_id`.
- `manifest_df`: ringkasan jumlah baris, kolom, domain, dan sumber file.
